# 12 — Investigation & Explainability

## Objective
Turn the frozen anomaly ranking into investigator-ready cases with evidence features, entity context, and interactive case inspection.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Score and rank cases

In [4]:
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
import joblib
fraud,_=load_data(); art=joblib.load(ART/"final_model.joblib"); 
X=art["history_builder"].transform(fraud); 
scores=-art["model"].decision_function(art["preprocessor"].transform(X[art["features"]])); 
X["anomaly_score"]=scores; ranked=X.sort_values("anomaly_score",ascending=False).copy(); 
display(ranked[["user_id","device_id","ip_address","purchase_value","purchase_time","anomaly_score","class"]].head(20))

,user_id,device_id,ip_address,purchase_value,purchase_time,anomaly_score,class
29639,335680,UOMHJMHDVTLAS,2.045463e+09,86,2015-04-26 19:18:33+00:00,0.247545,0
146768,257171,ZUSVMDEZRBDTX,1.502818e+09,47,2015-03-29 03:32:06+00:00,0.238457,0
8469,269203,KPAAACGRQWYIK,1.839748e+08,81,2015-04-21 06:57:47+00:00,0.233825,0
117952,120952,FFWAQIABHGYJC,1.955530e+08,11,2015-03-21 23:56:18+00:00,0.233364,0
150797,75490,RWZCXZTQUORQL,1.281304e+09,71,2015-01-04 17:22:39+00:00,0.233112,1
82342,102935,RWZCXZTQUORQL,1.281304e+09,71,2015-01-04 17:22:36+00:00,0.233112,1
18530,97115,RWZCXZTQUORQL,1.281304e+09,71,2015-01-04 17:22:27+00:00,0.233112,1
95727,27884,RWZCXZTQUORQL,1.281304e+09,71,2015-01-04 17:22:41+00:00,0.233112,1
2528,333354,RWZCXZTQUORQL,1.281304e+09,71,2015-01-04 17:22:38+00:00,0.233112,1
101902,66257,RWZCXZTQUORQL,1.281304e+09,71,2015-01-04 17:22:32+00:00,0.233112,1


## 2. Evidence attribution

In [5]:
ranked["value_z"]=abs(robust_z(ranked.purchase_value)); 
ranked["age_z"]=abs(robust_z(ranked.account_age_hours)); 
ev=["value_z","age_z","user_id_history_count","device_id_history_count","ip_address_history_count"]; 
ranked["top_evidence"]=ranked[ev].apply(lambda r:", ".join(r.sort_values(ascending=False).head(3).index),axis=1); 
display(ranked[["user_id","device_id","ip_address","purchase_value","account_age_hours","anomaly_score","top_evidence","class"]].head(50))

,user_id,device_id,ip_address,purchase_value,account_age_hours,anomaly_score,top_evidence,class
29639,335680,UOMHJMHDVTLAS,2.045463e+09,86,2619.385833,0.247545,"device_id_history_count, ip_address_history_co...",0
146768,257171,ZUSVMDEZRBDTX,1.502818e+09,47,1964.980000,0.238457,"device_id_history_count, ip_address_history_co...",0
8469,269203,KPAAACGRQWYIK,1.839748e+08,81,2514.794722,0.233825,"device_id_history_count, ip_address_history_co...",0
117952,120952,FFWAQIABHGYJC,1.955530e+08,11,1741.712778,0.233364,"device_id_history_count, ip_address_history_co...",0
150797,75490,RWZCXZTQUORQL,1.281304e+09,71,0.000278,0.233112,"device_id_history_count, ip_address_history_co...",1
82342,102935,RWZCXZTQUORQL,1.281304e+09,71,0.000278,0.233112,"device_id_history_count, ip_address_history_co...",1
18530,97115,RWZCXZTQUORQL,1.281304e+09,71,0.000278,0.233112,"device_id_history_count, ip_address_history_co...",1
95727,27884,RWZCXZTQUORQL,1.281304e+09,71,0.000278,0.233112,"device_id_history_count, ip_address_history_co...",1
2528,333354,RWZCXZTQUORQL,1.281304e+09,71,0.000278,0.233112,"device_id_history_count, ip_address_history_co...",1
101902,66257,RWZCXZTQUORQL,1.281304e+09,71,0.000278,0.233112,"device_id_history_count, ip_address_history_co...",1


## 3. Interactive case explorer

In [6]:
case=widgets.IntSlider(value=1,min=1,max=100,description="Case rank"); out=widgets.Output()
def inspect(*_):
 r=ranked.iloc[case.value-1]
 with out: out.clear_output(); 
 display(Markdown(f"## Rank {case.value} — score **{r.anomaly_score:.4f}**")); 
 display(pd.Series({k:r[k] for k in 
                    ["user_id","device_id","ip_address","purchase_value","purchase_time","account_age_hours","user_id_history_count","device_id_history_count","ip_address_history_count","class"]}).to_frame("value")); 
display(Markdown(f"**Evidence:** {r.top_evidence}"))
case.observe(inspect,'value'); 
display(case,out); inspect(); ranked.head(1000).to_parquet(ART/"investigation_ranked.parquet")

NameError: name 'r' is not defined